# Lesson 8: Time Travel - Rewinding the Graph

## Where We Are

In **Lesson 6** the checkpointer started saving State after every step - that gave us **memory**.
In **Lesson 7** we used those same checkpoints to **pause and resume** mid-execution.

But the checkpointer has been doing something more all along: it has been saving **every** intermediate state, not just the latest one. That means we have a full history of the graph's execution - and history we can replay from any point.

This lesson: how to **rewind** to a past checkpoint and **fork** an alternate execution from there.

---

## The Mental Model

> **Think of a video game with save points.** Every time the agent finishes a step, the checkpointer drops a save file. Normally you just play forward. But you can also reload **any** old save and play a different branch from that point - the original timeline is preserved, and a new one starts from where you reloaded.

Same agent. Same tools. Different question asked at a past decision point. Two timelines diverge.

---

## What We Will Build

We will reuse the **concierge agent from Lesson 6** (weather + population tools). Then:

| Step | What Happens |
|:---|:---|
| 1. Original timeline | Ask about Tokyo, then London. Build full history. |
| 2. Inspect history | List every checkpoint the agent went through. |
| 3. Pick a save point | Find the checkpoint just after the Tokyo answer. |
| 4. Fork | Resume from that checkpoint with a **different** second question. |
| 5. Compare | Original asked about London, fork asked about New York. Two histories now exist. |

---

## Graph Structure (Unchanged from Lesson 6)

```
         START
           |
           v
       +-------+
       | agent |<------+
       +-------+       |
           |           |
      should_use_tools |
        /        \     |
      END       tools  |
                  +----+
```

The graph itself does not change. What changes is **how we invoke it** - by passing a `checkpoint_id` in the config, we tell LangGraph: *"start from this past state, not from scratch."*

---

## Step 1: Setup

Same dual-environment auth pattern. Works in Colab or locally.

In [7]:
%pip install -q langgraph langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\sk72\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set"
print("API key loaded.")

API key loaded.


---

## Step 2: Define the Tools

Same fake weather and population lookups from Lesson 5/6.

In [9]:
from langchain_core.tools import tool

WEATHER_DATA = {
    "new york": "72F, Sunny",
    "london": "58F, Cloudy",
    "tokyo": "68F, Rainy",
}

POPULATION_DATA = {
    "new york": "8.3 million",
    "london": "8.9 million",
    "tokyo": "13.9 million",
}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return WEATHER_DATA.get(city.lower(), f"No weather data for {city}")

@tool
def get_population(city: str) -> str:
    """Get population of a city."""
    return POPULATION_DATA.get(city.lower(), f"No population data for {city}")

tools = [get_weather, get_population]

---

## Step 3: Build the Graph (Same as Lesson 6)

Concierge agent + checkpointer. Nothing new here - this is the foundation we will time-travel through.

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def agent(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

def should_use_tools(state: State) -> str:
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

workflow = StateGraph(State)
workflow.add_node("agent", agent)
workflow.add_node("tools", ToolNode(tools))
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_use_tools, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)
print("Graph compiled with checkpointer.")

Graph compiled with checkpointer.


---

## Step 4: Build the Original Timeline

Two questions on the same thread. The checkpointer is silently saving a snapshot after every node.

In [11]:
config = {"configurable": {"thread_id": "timeline-original"}}

print("--- Question 1 ---")
r1 = app.invoke({"messages": [("user", "What's the weather in Tokyo?")]}, config=config)
print(r1["messages"][-1].content)

print("\n--- Question 2 ---")
r2 = app.invoke({"messages": [("user", "What about the weather in London?")]}, config=config)
print(r2["messages"][-1].content)

--- Question 1 ---
The weather in Tokyo is currently 68°F and rainy.

--- Question 2 ---
The weather in London is currently 58°F and cloudy.


---

## Step 5: Inspect the History

`app.get_state_history(config)` returns every checkpoint LangGraph took, **newest first**. Each one is a `StateSnapshot` with:

| Field | Meaning |
|:---|:---|
| `config` | Contains the `checkpoint_id` - the unique save-file name |
| `values` | The full State at that moment |
| `next` | Which node(s) would run next from here |
| `metadata` | Step number, source, writes |

In [12]:
history = list(app.get_state_history(config))
print(f"Total checkpoints saved: {len(history)}\n")

for i, snap in enumerate(history):
    last_msg = snap.values["messages"][-1] if snap.values.get("messages") else None
    msg_type = type(last_msg).__name__ if last_msg else "-"
    next_node = ", ".join(snap.next) if snap.next else "(end)"
    n_msgs = len(snap.values.get("messages", []))
    print(f"[{i:2d}] next={next_node:20s} last={msg_type:15s} msgs={n_msgs}")

Total checkpoints saved: 10

[ 0] next=(end)                last=AIMessage       msgs=8
[ 1] next=agent                last=ToolMessage     msgs=7
[ 2] next=tools                last=AIMessage       msgs=6
[ 3] next=agent                last=HumanMessage    msgs=5
[ 4] next=__start__            last=AIMessage       msgs=4
[ 5] next=(end)                last=AIMessage       msgs=4
[ 6] next=agent                last=ToolMessage     msgs=3
[ 7] next=tools                last=AIMessage       msgs=2
[ 8] next=agent                last=HumanMessage    msgs=1
[ 9] next=__start__            last=-               msgs=0


Each line is one save point. Reading top to bottom is going **backwards in time** - from the final answer back to the empty starting state.

---

## Step 6: Pick a Save Point to Rewind To

We want the checkpoint **right after Question 1 finished** - meaning the Tokyo answer is in State, but Question 2 has not been asked yet.

That is the snapshot where:
- `next` is empty (the graph ended)
- The last message is the Tokyo answer (not the London one)

Looking at the history, that's the snapshot with **exactly 3 messages** (user question, AI tool call, tool result... wait - and the final AI answer = 4). Let's find it programmatically.

In [13]:
from langchain_core.messages import AIMessage

rewind_target = None
for snap in history:
    msgs = snap.values.get("messages", [])
    if not snap.next and msgs and isinstance(msgs[-1], AIMessage) and "tokyo" in msgs[-1].content.lower() and "london" not in msgs[-1].content.lower():
        rewind_target = snap
        break

print("Rewind target found:")
print(f"  checkpoint_id: {rewind_target.config['configurable']['checkpoint_id']}")
print(f"  messages at this point: {len(rewind_target.values['messages'])}")
print(f"  last message: {rewind_target.values['messages'][-1].content[:80]}")

Rewind target found:
  checkpoint_id: 1f157d1d-0ea0-6ad6-8003-281e828fd332
  messages at this point: 4
  last message: The weather in Tokyo is currently 68°F and rainy.


---

## Step 7: Fork the Timeline

To replay from a past checkpoint, pass its `config` (which carries the `checkpoint_id`) into `invoke`. LangGraph will:

1. Load the State as it was at that checkpoint
2. Apply your new input on top
3. Continue execution - creating new checkpoints that branch off the old timeline

This time we ask about **New York** instead of London.

In [14]:
fork_config = rewind_target.config

print("--- Alternate Question 2 (forked from after Tokyo) ---")
r_fork = app.invoke(
    {"messages": [("user", "What about the weather in New York?")]},
    config=fork_config,
)
print(r_fork["messages"][-1].content)

--- Alternate Question 2 (forked from after Tokyo) ---
The weather in New York is currently 72°F and sunny.


---

## Step 8: Verify Both Timelines Exist

The original asked about London. The fork asked about New York. The agent had the Tokyo context in both (proving the rewind worked - State was loaded from the save), but the second question diverged.

In [15]:
print("ORIGINAL TIMELINE - last user/assistant exchange:")
for m in r2["messages"][-2:]:
    print(f"  {type(m).__name__}: {m.content[:100]}")

print("\nFORKED TIMELINE - last user/assistant exchange:")
for m in r_fork["messages"][-2:]:
    print(f"  {type(m).__name__}: {m.content[:100]}")

ORIGINAL TIMELINE - last user/assistant exchange:
  ToolMessage: 58F, Cloudy
  AIMessage: The weather in London is currently 58°F and cloudy.

FORKED TIMELINE - last user/assistant exchange:
  ToolMessage: 72F, Sunny
  AIMessage: The weather in New York is currently 72°F and sunny.


Same starting context (Tokyo answer in history), two different continuations. That is time travel.

---

## Step 9: Editing the Past with `update_state`

Rewinding lets you replay from a past checkpoint. But what if you want to **change** what State looked like at that point before continuing?

`app.update_state(config, values, as_node=...)` writes new values into a checkpoint and returns a new config pointing at the modified snapshot. Then you can `invoke(None, config=new_config)` to continue from there.

The `as_node` parameter tells LangGraph *which node should be treated as having produced this update* - it controls where execution resumes from. Passing `START` makes it behave as if a fresh user turn just arrived, so the next step is `agent`.

Below: rewind to after Tokyo, **inject** a new user question into State (`as_node=START`), then continue.

In [ ]:
from langchain_core.messages import HumanMessage

edited_config = app.update_state(
    rewind_target.config,
    {"messages": [HumanMessage(content="Actually, compare the populations of Tokyo and London for me.")]},
    as_node=START,
)

print("--- Edited continuation ---")
r_edit = app.invoke(None, config=edited_config)
print(r_edit["messages"][-1].content)

AttributeError: 'HumanMessage' object has no attribute 'tool_calls'

Note `invoke(None, ...)` - we pass `None` as input because the new message is already in State from the `update_state` call. The graph just resumes from the modified checkpoint.

---

## Step 10: Visualize the Graph

Structurally identical to Lesson 6. Time travel is a **runtime capability** provided by the checkpointer - it does not change the graph itself.

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

---

## Summary

### What Changed from Lesson 7

| Lesson 7 (Human-in-the-Loop) | Lesson 8 (Time Travel) |
|:---|:---|
| Pause **during** execution, resume forward | Rewind **after** execution, restart from any past point |
| One timeline that pauses and continues | Multiple timelines forking from shared history |
| Uses `interrupt()` + `Command(resume=...)` | Uses `get_state_history()` + checkpoint config |

Both features ride on the **same** checkpointer from Lesson 6. Once you save state after every step, you can pause it, resume it, replay it, or rewrite it.

---

### The Time Travel Cycle

```
1. Run the graph normally       -> checkpoints accumulate
2. get_state_history(config)    -> list of save points (newest first)
3. Pick a snapshot              -> grab its .config (carries checkpoint_id)
4a. invoke(new_input, config)   -> fork from that point with new input
4b. update_state(config, vals)  -> rewrite State at that point
    then invoke(None, new_cfg)  -> continue from the rewritten state
```

---

### API Reference

| Component | Purpose |
|:---|:---|
| `app.get_state_history(config)` | Iterator of every checkpoint on this thread |
| `StateSnapshot.config` | Carries `checkpoint_id` - the address of this save point |
| `StateSnapshot.values` | Full State as it existed at that checkpoint |
| `StateSnapshot.next` | Tuple of nodes that would run next from here |
| `app.invoke(input, config_with_checkpoint_id)` | Fork - replay from that checkpoint |
| `app.update_state(config, values)` | Rewrite State at a checkpoint; returns new config |

---

### Key Takeaways

> 1. The checkpointer was always saving **every** step - not just the latest one
> 2. `get_state_history()` exposes that full save chain
> 3. Passing a past `checkpoint_id` in config tells `invoke` to start from that save
> 4. Forking does **not** erase the original - both timelines coexist in the checkpointer
> 5. `update_state()` lets you edit history before replaying it - useful for debugging or what-if analysis

---

**Next Lesson**: So far we have built one agent at a time. Lesson 9 introduces **multi-agent systems**: multiple specialized agents collaborating under a supervisor, with handoffs between them, producing emergent problem-solving behavior.